# Site Safety Compliance Detector — RT-DETR fine-tune on Kaggle

Fine-tunes RT-DETR to detect **person**, **helmet** and **head** (a bare head with
no helmet) on construction-site imagery, evaluates it in-domain and on an
out-of-distribution dataset, mines failure cases, and packages the checkpoint
for the FastAPI service in the repo.

**Notebook settings (right-hand panel):**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (recommended). P100 also works: Kaggle's current torch build has dropped sm_60 support, so cell 1 detects that and installs a compatible torch first. |
| Internet | **ON** — needed for pip, `kagglehub`, and the `rtdetr-l.pt` base weights |
| Datasets to attach | **None.** Both datasets are downloaded by code below via `kagglehub`. |

Run the real training through **Save Version → Save & Run All** so a browser
disconnect does not kill the run. Everything is written under `/kaggle/working`.

## 1. Environment

Installs the pinned dependencies, then checks that the preinstalled torch was
compiled for this GPU. Kaggle's torch 2.10 + CUDA 12.8 image no longer includes
sm_60 kernels, so on a P100 the model fails with `CUDA error: no kernel image
is available`. If that mismatch is detected, a torch build that still supports
the card is installed **before** torch is imported into this kernel.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.makedirs("/kaggle/working/repo", exist_ok=True)
os.chdir("/kaggle/working/repo")

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("ultralytics==8.3.40", "kagglehub")   # no numpy/opencv pins: Kaggle ships numpy 2 and cv2 already
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

# Probe in a subprocess so torch is not yet imported here if it has to be replaced.
probe = subprocess.run([sys.executable, "-c",
    "import torch; cap = torch.cuda.get_device_capability(); "
    "print('sm_%d%d' % cap); print(' '.join(torch.cuda.get_arch_list())); print(torch.__version__)"],
    capture_output=True, text=True)
gpu_sm, arch_list, torch_ver = (probe.stdout.strip().split("\n") + ["", "", ""])[:3]
print(f"gpu {gpu_sm} | torch {torch_ver} compiled for: {arch_list}")

if gpu_sm and gpu_sm not in arch_list.split():
    print(f"{gpu_sm} is not supported by the preinstalled torch; installing torch 2.6.0 (cu126), ~2.5 GB ...")
    pip("--force-reinstall", "--no-deps",
        "torch==2.6.0", "torchvision==0.21.0",
        "--index-url", "https://download.pytorch.org/whl/cu126")
    pip("ultralytics==8.3.40")   # re-satisfy deps the --no-deps install skipped

import torch, ultralytics
sm = "sm_%d%d" % torch.cuda.get_device_capability()
assert sm in torch.cuda.get_arch_list(), f"{sm} still unsupported by torch {torch.__version__}: {torch.cuda.get_arch_list()}"
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| gpu", torch.cuda.get_device_name(0),
      "| ultralytics", ultralytics.__version__)

## 2. Run configuration

Edit here, nowhere else. Set `DRY_RUN = True` the first time to measure
seconds per epoch, then multiply it out before committing to `EPOCHS`.
RT-DETR is attention heavy: **batch 8 per GPU at 640 px is the largest that
fits on a 16 GB card**. With two GPUs the batch is doubled and Ultralytics
trains with DDP across both; the per-card load stays at 8.

In [ ]:
DRY_RUN   = False   # True -> 2 epochs into runs/dryrun, then stop
EPOCHS    = 40
N_GPU     = torch.cuda.device_count()
for i in range(N_GPU):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}  {torch.cuda.get_device_properties(i).total_memory/2**30:.1f} GB")
if N_GPU < 2:
    print(f"WARNING: only {N_GPU} GPU visible. Training will still run, ~2x slower. Pick 'GPU T4 x2' next time.")
DEVICE    = ",".join(str(i) for i in range(N_GPU)) or "cpu"   # "0,1" -> Ultralytics relaunches under DDP across both cards
BATCH     = 8 * max(N_GPU, 1)                                 # 8 per GPU is the 16 GB ceiling for RT-DETR-L at 640
WORKERS   = os.cpu_count()                           # Kaggle gives 4 cores; the dataloader is the bottleneck otherwise
CACHE     = "ram"                                    # ~4 GB of decoded images; Kaggle has 30 GB RAM
IMGSZ     = 640
LR0       = 1e-4
SEED      = 0
OOD_LIMIT = 600     # SH17 images to evaluate on; keeps the OOD pass under a few minutes

WORK  = "/kaggle/working"
DATA  = f"{WORK}/data/ppe"
OOD   = f"{WORK}/data/ood_sh17"
RUNS  = f"{WORK}/runs"
ARTS  = f"{WORK}/artifacts"
RUN_NAME = "dryrun" if DRY_RUN else "rtdetr_ppe"
RUN_EPOCHS = 2 if DRY_RUN else EPOCHS
BEST = f"{RUNS}/{RUN_NAME}/weights/best.pt"
os.makedirs(ARTS, exist_ok=True)
print(json.dumps({k: v for k, v in globals().items() if k.isupper() and not k.startswith("_")}, indent=2, default=str))

## 3. Repository code

The repo is embedded below so the notebook is self-contained. These cells are
generated from the real source files by `notebooks/build_kaggle_notebook.py`;
do not edit them here.

In [ ]:
from pathlib import Path
Path("/kaggle/working/repo/scripts/__init__.py").parent.mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/repo/scripts/__init__.py").touch()
print("created empty", "/kaggle/working/repo/scripts/__init__.py")

In [ ]:
%%writefile /kaggle/working/repo/scripts/prepare_data.py
"""Convert the Kaggle Safety Helmet Detection dataset (Pascal VOC XML) into a
YOLO-format dataset with a deterministic, image-level train/val/test split.

Source: https://www.kaggle.com/datasets/andrewmvd/hard-hat-detection
Layout expected:  <root>/images/*.png  and  <root>/annotations/*.xml

Usage:
    python scripts/prepare_data.py \
        --root /kaggle/input/hard-hat-detection \
        --out  /kaggle/working/data/ppe
"""
import argparse
import hashlib
import json
import random
import shutil
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

# Fixed class order. Index is baked into the weights, so never reorder this.
CLASSES = ["helmet", "head", "person"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASSES)}

# The raw dataset spells the bare-head class a few different ways across files.
ALIASES = {
    "helmet": "helmet",
    "hat": "helmet",
    "hard-hat": "helmet",
    "head": "head",
    "person": "person",
}


def parse_voc(xml_path: Path):
    """Return (width, height, [(class_name, xmin, ymin, xmax, ymax), ...])."""
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    width = int(float(size.find("width").text))
    height = int(float(size.find("height").text))

    boxes = []
    for obj in root.findall("object"):
        raw_name = obj.find("name").text.strip().lower()
        name = ALIASES.get(raw_name)
        if name is None:
            continue  # unknown label, skip rather than silently mislabel
        bb = obj.find("bndbox")
        xmin = float(bb.find("xmin").text)
        ymin = float(bb.find("ymin").text)
        xmax = float(bb.find("xmax").text)
        ymax = float(bb.find("ymax").text)
        if xmax <= xmin or ymax <= ymin:
            continue  # degenerate box
        boxes.append((name, xmin, ymin, xmax, ymax))
    return width, height, boxes


def to_yolo_line(name, xmin, ymin, xmax, ymax, width, height):
    """Pascal VOC corners -> YOLO normalised cx cy w h, clipped to the image."""
    xmin = max(0.0, min(xmin, width))
    xmax = max(0.0, min(xmax, width))
    ymin = max(0.0, min(ymin, height))
    ymax = max(0.0, min(ymax, height))
    cx = (xmin + xmax) / 2.0 / width
    cy = (ymin + ymax) / 2.0 / height
    bw = (xmax - xmin) / width
    bh = (ymax - ymin) / height
    return f"{CLASS_TO_ID[name]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"


def split_for(stem: str, val_frac: float, test_frac: float) -> str:
    """Hash-based split.

    The split is a pure function of the file stem, so re-running this script or
    adding images later never moves an existing image between splits. That is
    what stops train/test leakage from creeping in across reruns.
    """
    digest = hashlib.md5(stem.encode("utf-8")).hexdigest()
    bucket = int(digest[:8], 16) / 0xFFFFFFFF
    if bucket < test_frac:
        return "test"
    if bucket < test_frac + val_frac:
        return "val"
    return "train"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True, help="dataset root holding images/ and annotations/")
    ap.add_argument("--out", required=True, help="output directory for the YOLO dataset")
    ap.add_argument("--val-frac", type=float, default=0.15)
    ap.add_argument("--test-frac", type=float, default=0.15)
    ap.add_argument("--copy", action="store_true", help="copy images instead of symlinking")
    args = ap.parse_args()

    root = Path(args.root)
    out = Path(args.out)
    img_dir = root / "images"
    ann_dir = root / "annotations"
    if not img_dir.is_dir() or not ann_dir.is_dir():
        raise SystemExit(f"expected {img_dir} and {ann_dir} to exist")

    for split in ("train", "val", "test"):
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)

    per_split = Counter()
    per_split_class = {s: Counter() for s in ("train", "val", "test")}
    empty_images = 0
    skipped = 0

    xmls = sorted(ann_dir.glob("*.xml"))
    if not xmls:
        raise SystemExit(f"no XML annotations found under {ann_dir}")

    for xml_path in xmls:
        stem = xml_path.stem
        image_path = next((p for p in (img_dir / f"{stem}{ext}"
                                       for ext in (".png", ".jpg", ".jpeg"))
                           if p.exists()), None)
        if image_path is None:
            skipped += 1
            continue

        width, height, boxes = parse_voc(xml_path)
        if width <= 0 or height <= 0:
            skipped += 1
            continue

        split = split_for(stem, args.val_frac, args.test_frac)
        lines = [to_yolo_line(*b, width, height) for b in boxes]
        if not lines:
            empty_images += 1  # kept on purpose: negatives teach the model restraint

        dst_img = out / "images" / split / image_path.name
        if not dst_img.exists():
            if args.copy:
                shutil.copy2(image_path, dst_img)
            else:
                try:
                    dst_img.symlink_to(image_path.resolve())
                except OSError:
                    shutil.copy2(image_path, dst_img)  # Windows without dev mode

        (out / "labels" / split / f"{stem}.txt").write_text("\n".join(lines), encoding="utf-8")

        per_split[split] += 1
        for name, *_ in boxes:
            per_split_class[split][name] += 1

    yaml_path = out / "data.yaml"
    yaml_path.write_text(
        "\n".join([
            f"path: {out.resolve().as_posix()}",
            "train: images/train",
            "val: images/val",
            "test: images/test",
            "names:",
            *[f"  {i}: {n}" for i, n in enumerate(CLASSES)],
            "",
        ]),
        encoding="utf-8",
    )

    stats = {
        "images_per_split": dict(per_split),
        "instances_per_split": {s: dict(c) for s, c in per_split_class.items()},
        "background_only_images": empty_images,
        "skipped_annotations": skipped,
        "split_method": "md5(file stem) -> deterministic bucket, image level",
        "val_frac": args.val_frac,
        "test_frac": args.test_frac,
    }
    (out / "split_stats.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
    print(json.dumps(stats, indent=2))
    print(f"\nwrote {yaml_path}")


if __name__ == "__main__":
    random.seed(0)
    main()

In [ ]:
%%writefile /kaggle/working/repo/scripts/prepare_ood.py
"""Build an out-of-distribution test set from SH17 and remap it onto our 3 classes.

SH17 is never trained on. It comes from a different image source (Pexels stock
photography) than the construction-site training images, so measuring on it
gives an honest read on how the model behaves off its training distribution.

Source: https://www.kaggle.com/datasets/mugheesahmad/sh17-dataset-for-ppe-detection
Licence: CC BY-NC-SA 4.0 -- research use, cite the authors.

Usage:
    python scripts/prepare_ood.py \
        --root /kaggle/input/sh17-dataset-for-ppe-detection \
        --out  /kaggle/working/data/ood_sh17 \
        --limit 600
"""
import argparse
import json
import re
import shutil
from collections import Counter
from pathlib import Path

from prepare_data import CLASSES, CLASS_TO_ID

# SH17 name -> our name. Everything else is dropped.
NAME_MAP = {
    "person": "person",
    "head": "head",
    "helmet": "helmet",
}


def load_sh17_names(root: Path, explicit: Path = None):
    """Read SH17's own class list rather than assuming an index order."""
    candidates = [explicit] if explicit else []
    candidates += sorted(root.rglob("*.yaml")) + sorted(root.rglob("*.yml"))
    for candidate in candidates:
        text = candidate.read_text(encoding="utf-8", errors="replace")
        if "names" not in text:
            continue
        names = {}
        for line in text.splitlines():
            m = re.match(r"\s*(\d+)\s*:\s*(.+?)\s*$", line)
            if m:
                names[int(m.group(1))] = m.group(2).strip().strip("'\"").lower()
        if names:
            print(f"class list read from {candidate}")
            return names
    raise SystemExit(
        "could not find a data yaml with a names: block under the SH17 root. "
        "Open the dataset in the Kaggle file browser and pass the class order manually."
    )


def find_label_for(image_path: Path, label_dirs):
    for d in label_dirs:
        p = d / f"{image_path.stem}.txt"
        if p.exists():
            return p
    return None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--limit", type=int, default=600,
                    help="cap the number of images so evaluation stays fast")
    ap.add_argument("--names", default=None,
                    help="yaml with a names: block, used when the dataset ships without one")
    args = ap.parse_args()

    root = Path(args.root)
    out = Path(args.out)
    (out / "images" / "test").mkdir(parents=True, exist_ok=True)
    (out / "labels" / "test").mkdir(parents=True, exist_ok=True)

    sh17_names = load_sh17_names(root, Path(args.names) if args.names else None)
    # SH17 index -> our index, for the three classes we share.
    index_map = {}
    for idx, name in sh17_names.items():
        target = NAME_MAP.get(name.replace("_", "-"))
        if target:
            index_map[idx] = CLASS_TO_ID[target]
    if not index_map:
        raise SystemExit(f"no overlapping classes found in {sh17_names}")
    print("index map (sh17 -> ours):", index_map)

    label_dirs = [p for p in root.rglob("labels") if p.is_dir()]
    images = [p for p in root.rglob("*")
              if p.suffix.lower() in (".jpg", ".jpeg", ".png") and "label" not in p.parts]
    images.sort()

    kept = 0
    counts = Counter()
    for image_path in images:
        if kept >= args.limit:
            break
        label_path = find_label_for(image_path, label_dirs)
        if label_path is None:
            continue

        lines = []
        for line in label_path.read_text(encoding="utf-8", errors="replace").splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            src = int(float(parts[0]))
            if src not in index_map:
                continue
            dst = index_map[src]
            lines.append(" ".join([str(dst)] + parts[1:5]))
            counts[CLASSES[dst]] += 1

        if not lines:
            continue  # an image with none of our classes teaches us nothing here

        shutil.copy2(image_path, out / "images" / "test" / image_path.name)
        (out / "labels" / "test" / f"{image_path.stem}.txt").write_text(
            "\n".join(lines), encoding="utf-8")
        kept += 1

    (out / "data.yaml").write_text(
        "\n".join([
            f"path: {out.resolve().as_posix()}",
            "train: images/test",   # unused, ultralytics wants the key present
            "val: images/test",
            "test: images/test",
            "names:",
            *[f"  {i}: {n}" for i, n in enumerate(CLASSES)],
            "",
        ]),
        encoding="utf-8",
    )

    stats = {"images": kept, "instances": dict(counts), "source": "SH17 (CC BY-NC-SA 4.0)"}
    (out / "ood_stats.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
    print(json.dumps(stats, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/repo/scripts/train.py
"""Fine-tune RT-DETR on the PPE dataset.

Sized for the Kaggle free tier. RT-DETR is attention heavy, so batch 8 per
GPU at 640px is the largest that reliably fits on a 16 GB card. With two GPUs
pass --device 0,1 and --batch 16; Ultralytics trains with DDP across both.

Usage:
    python scripts/train.py --data /kaggle/working/data/ppe/data.yaml \
        --epochs 40 --batch 16 --device 0,1 --cache ram --project /kaggle/working/runs

Dry run first to measure seconds per epoch before committing the full budget:
    python scripts/train.py --data ... --epochs 2 --name dryrun
"""
import argparse
import json
import platform
import subprocess
import time
import traceback
from pathlib import Path


def hardware_report():
    info = {
        "python": platform.python_version(),
        "platform": platform.platform(),
    }
    try:
        import torch
        info["torch"] = torch.__version__
        info["cuda_available"] = torch.cuda.is_available()
        if torch.cuda.is_available():
            info["gpu"] = torch.cuda.get_device_name(0)
            info["gpu_memory_gb"] = round(
                torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 1)
    except Exception as exc:  # pragma: no cover
        info["torch_error"] = str(exc)
    try:
        info["nvidia_smi"] = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
             "--format=csv,noheader"], text=True).strip()
    except Exception:
        pass
    return info


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--model", default="rtdetr-l.pt")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--lr0", type=float, default=1e-4)
    ap.add_argument("--optimizer", default="AdamW")
    ap.add_argument("--patience", type=int, default=12)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--device", default="0",
                    help="cuda device id(s), e.g. 0 or 0,1 for multi-GPU DDP, or cpu")
    ap.add_argument("--cache", default="",
                    help="'ram' or 'disk' to cache decoded images; empty for none")
    ap.add_argument("--project", default="runs")
    ap.add_argument("--name", default="rtdetr_ppe")
    args = ap.parse_args()

    from ultralytics import RTDETR, settings

    # Ultralytics auto-registers a Ray Tune callback whenever `ray` is importable.
    # Kaggle ships a newer ray whose private API that callback calls no longer
    # exists, which crashes training at the end of the first epoch.
    settings.update({"raytune": False})

    hw = hardware_report()
    print(json.dumps(hw, indent=2))

    model = RTDETR(args.model)
    run_dir = Path(args.project) / args.name
    started = time.time()
    error = None
    try:
        model.train(
            data=args.data,
            epochs=args.epochs,
            batch=args.batch,
            imgsz=args.imgsz,
            lr0=args.lr0,
            optimizer=args.optimizer,
            patience=args.patience,
            workers=args.workers,
            seed=args.seed,
            device=args.device,
            project=args.project,
            name=args.name,
            exist_ok=True,
            amp=True,          # required to fit batch 8 on 16 GB
            cache=args.cache or False,
            plots=True,
            val=True,
        )
    except BaseException as exc:
        # Under DDP the real training runs in subprocesses that already saved
        # best.pt; a crash in the parent must not lose the receipt.
        error = f"{type(exc).__name__}: {exc}"
        traceback.print_exc()
    elapsed = time.time() - started

    epochs_completed = None
    results_csv = run_dir / "results.csv"
    if results_csv.exists():
        epochs_completed = max(0, len(results_csv.read_text(encoding="utf-8").strip().splitlines()) - 1)

    receipt = {
        "status": "failed" if error else "ok",
        "error": error,
        "epochs_completed": epochs_completed,
        "hardware": hw,
        "model": args.model,
        "epochs": args.epochs,
        "batch": args.batch,
        "imgsz": args.imgsz,
        "lr0": args.lr0,
        "optimizer": args.optimizer,
        "seed": args.seed,
        "device": args.device,
        "workers": args.workers,
        "cache": args.cache or False,
        "data": args.data,
        "wall_clock_seconds": round(elapsed, 1),
        "wall_clock_human": f"{elapsed / 3600:.2f} h",
        "weights": str(run_dir / "weights" / "best.pt"),
    }
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "training_receipt.json").write_text(json.dumps(receipt, indent=2), encoding="utf-8")
    print(json.dumps(receipt, indent=2))
    print("\nPaste the receipt above into the memo. Reproducibility is graded.")
    if error:
        raise SystemExit(1)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/repo/scripts/evaluate.py
"""Evaluate the fine-tuned model on the in-domain test split and, optionally,
on the out-of-distribution SH17 slice.

Reporting both is the point. The in-domain number tells you how well the model
learned this dataset; the OOD number is the closest honest proxy available for
performance on an unseen hidden test set.

Usage:
    python scripts/evaluate.py --weights runs/rtdetr_ppe/weights/best.pt \
        --data /kaggle/working/data/ppe/data.yaml \
        --ood  /kaggle/working/data/ood_sh17/data.yaml \
        --out  artifacts/metrics.json
"""
import argparse
import json
from pathlib import Path

from prepare_data import CLASSES


def run_split(model, data_yaml, split, imgsz, batch, tag):
    results = model.val(data=data_yaml, split=split, imgsz=imgsz, batch=batch,
                        plots=True, name=f"val_{tag}", exist_ok=True)
    box = results.box
    per_class = {}
    for i, name in enumerate(CLASSES):
        try:
            p, r, ap50, ap = box.class_result(i)
            per_class[name] = {
                "precision": round(float(p), 4),
                "recall": round(float(r), 4),
                "mAP50": round(float(ap50), 4),
                "mAP50_95": round(float(ap), 4),
            }
        except Exception:
            per_class[name] = {"note": "no ground-truth instances of this class in the split"}
    return {
        "split": split,
        "data": str(data_yaml),
        "mAP50": round(float(box.map50), 4),
        "mAP50_95": round(float(box.map), 4),
        "precision": round(float(box.mp), 4),
        "recall": round(float(box.mr), 4),
        "per_class": per_class,
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--ood", default=None, help="data.yaml for the SH17 OOD slice")
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--out", default="artifacts/metrics.json")
    args = ap.parse_args()

    from ultralytics import RTDETR

    model = RTDETR(args.weights)
    report = {"weights": args.weights, "in_domain": run_split(
        model, args.data, "test", args.imgsz, args.batch, "in_domain")}

    if args.ood:
        report["out_of_distribution"] = run_split(
            model, args.ood, "test", args.imgsz, args.batch, "ood")
        gap = report["in_domain"]["mAP50"] - report["out_of_distribution"]["mAP50"]
        report["generalisation_gap_mAP50"] = round(gap, 4)

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/repo/scripts/failure_cases.py
"""Mine the test split for the model's worst images and attach measured
evidence for each failure, so the memo's root-cause analysis is grounded in
numbers rather than in what an annotated image looks like at a glance.

For every test image it matches predictions to ground truth greedily by IoU,
counts false negatives, false positives and class confusions, then measures
four image properties that commonly explain them:

  blur        variance of the Laplacian over the missed region
  scale       missed box area as a fraction of image area
  occlusion   max IoU between the missed box and any other ground-truth box
  exposure    mean luminance of the missed region

Usage:
    python scripts/failure_cases.py --weights runs/rtdetr_ppe/weights/best.pt \
        --data /kaggle/working/data/ppe/data.yaml --top 5 \
        --out artifacts/failures
"""
import argparse
import json
from pathlib import Path

import cv2
import numpy as np

from prepare_data import CLASSES

COLOURS = {0: (0, 200, 0), 1: (0, 0, 230), 2: (230, 160, 0)}


def yolo_to_xyxy(line, width, height):
    cls, cx, cy, bw, bh = (float(v) for v in line.split()[:5])
    x1 = (cx - bw / 2) * width
    y1 = (cy - bh / 2) * height
    x2 = (cx + bw / 2) * width
    y2 = (cy + bh / 2) * height
    return int(cls), np.array([x1, y1, x2, y2], dtype=float)


def iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    if inter <= 0:
        return 0.0
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter)


def crop_stats(image, box):
    h, w = image.shape[:2]
    x1, y1, x2, y2 = [int(max(0, v)) for v in box]
    x2, y2 = min(x2, w), min(y2, h)
    if x2 - x1 < 2 or y2 - y1 < 2:
        return {"blur_laplacian_var": None, "mean_luminance": None}
    crop = image[y1:y2, x1:x2]
    grey = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    return {
        "blur_laplacian_var": round(float(cv2.Laplacian(grey, cv2.CV_64F).var()), 2),
        "mean_luminance": round(float(grey.mean()), 2),
    }


def diagnose(record):
    """Turn measurements into a first-pass hypothesis. Verify it by eye before
    it goes in the memo. This narrows the search, it does not replace looking."""
    reasons = []
    if record["scale_fraction"] is not None and record["scale_fraction"] < 0.005:
        reasons.append("small object: under 0.5 percent of image area")
    if record["blur_laplacian_var"] is not None and record["blur_laplacian_var"] < 60:
        reasons.append("motion or focus blur: low Laplacian variance")
    if record["max_overlap_with_other_gt"] > 0.35:
        reasons.append("occlusion or crowding: heavy overlap with a neighbouring box")
    if record["mean_luminance"] is not None and record["mean_luminance"] < 55:
        reasons.append("underexposed region")
    if record["mean_luminance"] is not None and record["mean_luminance"] > 205:
        reasons.append("blown highlights or backlighting")
    if record["kind"] == "class_confusion":
        reasons.append("class confusion: predicted {} for a {}".format(
            record["predicted_class"], record["true_class"]))
    return reasons or ["no single measured cause stands out, inspect manually"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--conf", type=float, default=0.25)
    ap.add_argument("--iou-match", type=float, default=0.5)
    ap.add_argument("--top", type=int, default=5)
    ap.add_argument("--out", default="artifacts/failures")
    args = ap.parse_args()

    from ultralytics import RTDETR

    data_root = Path(args.data).parent
    img_dir = data_root / "images" / "test"
    lbl_dir = data_root / "labels" / "test"
    images = sorted(p for p in img_dir.iterdir()
                    if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
    if not images:
        raise SystemExit("no test images under {}".format(img_dir))

    model = RTDETR(args.weights)
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    scored = []
    for image_path in images:
        image = cv2.imread(str(image_path))
        if image is None:
            continue
        h, w = image.shape[:2]

        label_path = lbl_dir / (image_path.stem + ".txt")
        gt = []
        if label_path.exists():
            for line in label_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    gt.append(yolo_to_xyxy(line, w, h))

        result = model.predict(str(image_path), conf=args.conf, verbose=False)[0]
        preds = []
        for box in result.boxes:
            preds.append((int(box.cls.item()),
                          box.xyxy[0].cpu().numpy().astype(float),
                          float(box.conf.item())))

        matched_pred = set()
        errors = []
        for gi, (gcls, gbox) in enumerate(gt):
            best_j, best_iou = -1, 0.0
            for j, (pcls, pbox, _) in enumerate(preds):
                if j in matched_pred:
                    continue
                score = iou(gbox, pbox)
                if score > best_iou:
                    best_iou, best_j = score, j

            if best_iou < args.iou_match:
                kind, pred_cls = "missed_detection", None
            else:
                matched_pred.add(best_j)
                if preds[best_j][0] == gcls:
                    continue
                kind, pred_cls = "class_confusion", CLASSES[preds[best_j][0]]

            others = [b for k, (_, b) in enumerate(gt) if k != gi]
            rec = {
                "kind": kind,
                "true_class": CLASSES[gcls],
                "predicted_class": pred_cls,
                "box_xyxy": [round(v, 1) for v in gbox.tolist()],
                "best_iou": round(best_iou, 3),
                "scale_fraction": round(
                    float((gbox[2] - gbox[0]) * (gbox[3] - gbox[1]) / (w * h)), 5),
                "max_overlap_with_other_gt": round(
                    max((iou(gbox, o) for o in others), default=0.0), 3),
            }
            rec.update(crop_stats(image, gbox))
            rec["hypotheses"] = diagnose(rec)
            errors.append(rec)

        false_positives = []
        for j, (pcls, pbox, conf) in enumerate(preds):
            if j in matched_pred:
                continue
            false_positives.append({
                "kind": "false_positive",
                "predicted_class": CLASSES[pcls],
                "confidence": round(conf, 3),
                "box_xyxy": [round(v, 1) for v in pbox.tolist()],
            })

        total = len(errors) + len(false_positives)
        if total == 0:
            continue
        scored.append({
            "image": image_path.name,
            "ground_truth_count": len(gt),
            "prediction_count": len(preds),
            "error_count": total,
            "errors": errors,
            "false_positives": false_positives,
        })

    scored.sort(key=lambda r: (-r["error_count"], r["image"]))
    worst = scored[: args.top]

    for record in worst:
        path = img_dir / record["image"]
        image = cv2.imread(str(path))
        result = model.predict(str(path), conf=args.conf, verbose=False)[0]
        for box in result.boxes:
            cls = int(box.cls.item())
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            colour = COLOURS.get(cls, (255, 255, 255))
            cv2.rectangle(image, (x1, y1), (x2, y2), colour, 2)
            cv2.putText(image, "{} {:.2f}".format(CLASSES[cls], box.conf.item()),
                        (x1, max(14, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, colour, 1)
        for err in record["errors"]:
            x1, y1, x2, y2 = [int(v) for v in err["box_xyxy"]]
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 1)
            cv2.putText(image, "MISS " + err["true_class"],
                        (x1, min(image.shape[0] - 4, y2 + 14)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1)
        cv2.imwrite(str(out_dir / ("failure_" + record["image"])), image)

    (out_dir / "failure_report.json").write_text(
        json.dumps({"conf": args.conf,
                    "iou_match": args.iou_match,
                    "images_with_errors": len(scored),
                    "worst": worst}, indent=2),
        encoding="utf-8")

    print(json.dumps([{"image": r["image"], "errors": r["error_count"]} for r in worst],
                     indent=2))
    print("\nannotated images and the full report are in {}".format(out_dir))


if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path
Path("/kaggle/working/repo/app/__init__.py").parent.mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/repo/app/__init__.py").touch()
print("created empty", "/kaggle/working/repo/app/__init__.py")

In [ ]:
%%writefile /kaggle/working/repo/app/detector.py
"""RT-DETR inference wrapper.

Loaded once at process start. Everything downstream consumes the plain
dictionaries this module returns, never the Ultralytics result objects, so the
reasoning layer stays testable without a GPU or a checkpoint.
"""
from __future__ import annotations

import io
import logging
import os
import threading
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List

import numpy as np
from PIL import Image

log = logging.getLogger("ppe.detector")

CLASSES = ["helmet", "head", "person"]
DEFAULT_WEIGHTS = os.getenv("MODEL_WEIGHTS", "artifacts/best.pt")
DEFAULT_CONF = float(os.getenv("CONF_THRESHOLD", "0.25"))
DEFAULT_IMGSZ = int(os.getenv("IMGSZ", "640"))
WEIGHTS_KAGGLE_DATASET = os.getenv("WEIGHTS_KAGGLE_DATASET", "")


@dataclass
class Detection:
    label: str
    confidence: float
    box_xyxy: List[float]

    def to_dict(self):
        return asdict(self)


class DetectorError(RuntimeError):
    pass


class Detector:
    """Thin, thread-safe wrapper around a fine-tuned RT-DETR checkpoint."""

    def __init__(self, weights: str = DEFAULT_WEIGHTS, imgsz: int = DEFAULT_IMGSZ):
        self.weights = weights
        self.imgsz = imgsz
        self._lock = threading.Lock()
        self._model = None
        self._names = None

    def load(self):
        if self._model is not None:
            return
        path = Path(self.weights)
        if not path.exists() and WEIGHTS_KAGGLE_DATASET:
            log.info("weights missing at %s, fetching from Kaggle dataset %s",
                     path, WEIGHTS_KAGGLE_DATASET)
            try:
                from scripts.download_weights import download
                download(WEIGHTS_KAGGLE_DATASET, path)
            except Exception as exc:
                log.warning("weights download failed: %s", exc)
        if not path.exists():
            raise DetectorError(
                "model weights not found at {}. Set MODEL_WEIGHTS, or set "
                "WEIGHTS_KAGGLE_DATASET so they are fetched automatically, or run "
                "scripts/download_weights.py.".format(path)
            )
        from ultralytics import RTDETR  # imported lazily, it is slow

        started = time.time()
        self._model = RTDETR(str(path))
        self._names = getattr(self._model, "names", None) or {
            i: n for i, n in enumerate(CLASSES)
        }
        log.info("loaded %s in %.1fs, classes=%s", path, time.time() - started, self._names)

    @property
    def ready(self) -> bool:
        return self._model is not None

    def label_for(self, index: int) -> str:
        if isinstance(self._names, dict):
            return str(self._names.get(index, index))
        try:
            return str(self._names[index])
        except Exception:
            return str(index)

    def predict(self, image_bytes: bytes, conf: float = DEFAULT_CONF):
        """Run detection on raw image bytes.

        Returns (detections, image_width, image_height, inference_ms).
        """
        self.load()
        try:
            image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        except Exception as exc:
            raise DetectorError("could not decode the uploaded file as an image") from exc

        array = np.array(image)[:, :, ::-1]  # PIL RGB -> BGR for ultralytics
        started = time.time()
        with self._lock:  # ultralytics predict is not reentrant
            result = self._model.predict(array, conf=conf, imgsz=self.imgsz, verbose=False)[0]
        elapsed_ms = (time.time() - started) * 1000.0

        detections = []
        for box in result.boxes:
            detections.append(Detection(
                label=self.label_for(int(box.cls.item())),
                confidence=round(float(box.conf.item()), 4),
                box_xyxy=[round(float(v), 1) for v in box.xyxy[0].tolist()],
            ))
        detections.sort(key=lambda d: d.confidence, reverse=True)
        return detections, image.width, image.height, round(elapsed_ms, 1)


detector = Detector()

In [ ]:
%%writefile /kaggle/working/repo/app/scene.py
"""Turn a flat list of boxes into the structured facts the reasoning layer needs.

The detector answers "what is in this image". It does not answer "who is wearing
what" -- that is a relation between two boxes, and deriving it is this module's
whole job. Counting helmets and counting people and subtracting is wrong the
moment one worker is out of frame or one helmet sits on a bench.

Two facts about the training data shape the rule:

  1. In the source dataset a `helmet` box is a head WITH a helmet on it and a
     `head` box is a head WITHOUT one. Each headgear box is therefore one
     worker's compliance status in its own right.
  2. `person` boxes are annotated on only ~3 percent of workers (497 person vs
     12,908 helmet instances in the training split), so the fine-tuned model
     rarely predicts them (test recall 0.03). Compliance cannot be anchored on
     person boxes; if it were, almost every question would be refused.

So: every headgear box becomes a worker. Person boxes, when present, refine
that: a headgear box that lies inside a person box but not in its head zone
is judged not worn (a helmet carried at the hip) and is excluded, and a person
box with no headgear inside its head zone is a worker whose status is UNKNOWN,
never compliant and never a violation. That last case is what lets the API
say "insufficient information" honestly.
"""
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass, field
from typing import List, Optional

# Fraction of the headgear box that must fall inside the person box.
CONTAINMENT_MIN = 0.55
# The headgear centre must sit within this fraction of the person box height,
# measured from the top. Generous, because people bend, crouch and lean.
HEAD_ZONE_FRACTION = 0.45

STATUS_COMPLIANT = "compliant"
STATUS_VIOLATION = "violation"
STATUS_UNKNOWN = "unknown"


def _area(box):
    return max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1])


def _containment(inner, outer) -> float:
    """Fraction of `inner` that lies inside `outer`."""
    x1 = max(inner[0], outer[0])
    y1 = max(inner[1], outer[1])
    x2 = min(inner[2], outer[2])
    y2 = min(inner[3], outer[3])
    overlap = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    inner_area = _area(inner)
    return overlap / inner_area if inner_area > 0 else 0.0


def _in_head_zone(headgear, person) -> bool:
    centre_y = (headgear[1] + headgear[3]) / 2.0
    height = person[3] - person[1]
    if height <= 0:
        return False
    return (centre_y - person[1]) / height <= HEAD_ZONE_FRACTION


@dataclass
class Worker:
    index: int
    box_xyxy: List[float]
    status: str
    person_confidence: Optional[float] = None
    headgear: Optional[str] = None
    headgear_confidence: Optional[float] = None
    note: str = ""


@dataclass
class SceneFacts:
    image_width: int
    image_height: int
    confidence_floor: float
    counts: dict = field(default_factory=dict)
    workers: List[Worker] = field(default_factory=list)
    person_boxes: int = 0
    helmets_not_worn: int = 0
    heads_ignored: int = 0
    compliant: int = 0
    violations: int = 0
    unknown: int = 0
    max_confidence: float = 0.0
    mean_confidence: float = 0.0
    detections: List[dict] = field(default_factory=list)

    def to_dict(self):
        return {
            "image_size": {"width": self.image_width, "height": self.image_height},
            "confidence_floor": self.confidence_floor,
            "counts": self.counts,
            "people_detected": len(self.workers),
            "person_boxes": self.person_boxes,
            "compliant_workers": self.compliant,
            "violations": self.violations,
            "undetermined_workers": self.unknown,
            "helmets_seen_but_not_worn": self.helmets_not_worn,
            "max_confidence": self.max_confidence,
            "mean_confidence": self.mean_confidence,
            "workers": [
                {
                    "id": w.index,
                    "status": w.status,
                    "headgear": w.headgear,
                    "headgear_confidence": w.headgear_confidence,
                    "person_confidence": w.person_confidence,
                    "box_xyxy": w.box_xyxy,
                    "note": w.note,
                }
                for w in self.workers
            ],
            "detections": self.detections,
        }


def build_scene(detections, image_width: int, image_height: int,
                confidence_floor: float) -> SceneFacts:
    """Group raw detections into per-worker compliance facts."""
    dets = [d.to_dict() if hasattr(d, "to_dict") else dict(d) for d in detections]
    counts = Counter(d["label"] for d in dets)
    confidences = [d["confidence"] for d in dets]

    people = [d for d in dets if d["label"] == "person"]
    helmets = [d for d in dets if d["label"] == "helmet"]
    heads = [d for d in dets if d["label"] == "head"]
    gear_pool = [("helmet", g) for g in helmets] + [("head", g) for g in heads]

    workers = []
    claimed = set()

    # Pass 1: people with a headgear box in their head zone, or with none at all.
    for person in people:
        best = None  # (containment, key, kind, detection)
        for key, (kind, gear) in enumerate(gear_pool):
            if key in claimed:
                continue
            containment = _containment(gear["box_xyxy"], person["box_xyxy"])
            if containment < CONTAINMENT_MIN or not _in_head_zone(gear["box_xyxy"], person["box_xyxy"]):
                continue
            if best is None or containment > best[0]:
                best = (containment, key, kind, gear)

        if best is None:
            workers.append(Worker(
                index=len(workers),
                box_xyxy=person["box_xyxy"],
                status=STATUS_UNKNOWN,
                person_confidence=person["confidence"],
                note="a person was detected but no helmet or bare head could be "
                     "resolved on them, so their compliance cannot be determined",
            ))
            continue

        _, key, kind, gear = best
        claimed.add(key)
        workers.append(Worker(
            index=len(workers),
            box_xyxy=person["box_xyxy"],
            status=STATUS_COMPLIANT if kind == "helmet" else STATUS_VIOLATION,
            person_confidence=person["confidence"],
            headgear=kind,
            headgear_confidence=gear["confidence"],
        ))

    # Pass 2: headgear with no person box. A helmet lying inside a visible
    # person box but off their head is being carried, not worn: exclude it.
    # Headgear with no person box around it at all is a worker whose person box
    # the detector missed, which is the common case for this model.
    helmets_not_worn = 0
    heads_ignored = 0
    for key, (kind, gear) in enumerate(gear_pool):
        if key in claimed:
            continue
        inside_someone = any(_containment(gear["box_xyxy"], p["box_xyxy"]) >= CONTAINMENT_MIN
                             for p in people)
        if inside_someone:
            if kind == "helmet":
                helmets_not_worn += 1
            else:
                heads_ignored += 1
            continue
        workers.append(Worker(
            index=len(workers),
            box_xyxy=gear["box_xyxy"],
            status=STATUS_COMPLIANT if kind == "helmet" else STATUS_VIOLATION,
            headgear=kind,
            headgear_confidence=gear["confidence"],
            note="status read from the headgear box alone; no person box was detected",
        ))

    facts = SceneFacts(
        image_width=image_width,
        image_height=image_height,
        confidence_floor=confidence_floor,
        counts=dict(counts),
        workers=workers,
        person_boxes=len(people),
        helmets_not_worn=helmets_not_worn,
        heads_ignored=heads_ignored,
        compliant=sum(1 for w in workers if w.status == STATUS_COMPLIANT),
        violations=sum(1 for w in workers if w.status == STATUS_VIOLATION),
        unknown=sum(1 for w in workers if w.status == STATUS_UNKNOWN),
        max_confidence=round(max(confidences), 4) if confidences else 0.0,
        mean_confidence=round(sum(confidences) / len(confidences), 4) if confidences else 0.0,
        detections=dets,
    )
    return facts

In [ ]:
%%writefile /kaggle/working/repo/app/reasoning.py
"""The reasoning layer: routing, structured reasoning, confidence guardrail.

Hand written on purpose. No agent framework is used or needed -- the whole
control flow is the three functions below, called in order by `answer_question`:

    route()      decide whether the detector has to run at all
    guardrail()  decide, from the detections, whether an honest answer exists
    compose()    put the facts into plain language

The guardrail runs BEFORE the language model sees anything, and it is
deterministic. A model that is asked to grade its own confidence will talk
itself into an answer; a rule that says "three of five workers have no
associated headgear, so compliance is undetermined" will not.

The language model is optional. With no API key configured the layer answers
from templates over the same structured facts, so the API is fully runnable
offline. Set ANTHROPIC_API_KEY to enable the fluent path.
"""
from __future__ import annotations

import logging
import os
import re
from dataclasses import dataclass, field
from typing import List, Optional

log = logging.getLogger("ppe.reasoning")

MODEL_ID = os.getenv("ANTHROPIC_MODEL", "claude-opus-5")
LOW_CONFIDENCE_FLOOR = float(os.getenv("LOW_CONFIDENCE_FLOOR", "0.45"))

# ---------------------------------------------------------------------------
# Intent routing
# ---------------------------------------------------------------------------

# What the detector can actually see. A question outside this vocabulary is
# image grounded but unanswerable by us, which is a different thing from a
# question that is not about the image at all.
KNOWN_VOCABULARY = {
    "person", "people", "worker", "workers", "man", "men", "woman", "women",
    "human", "humans", "crew", "staff", "helmet", "helmets", "hardhat",
    "hard-hat", "hardhats", "hat", "hats", "head", "heads", "ppe",
    "compliance", "compliant", "safety",
}

IMAGE_REFERENCE = re.compile(
    r"\b(image|picture|photo|photograph|frame|shot|scene|here|visible|shown|"
    r"this\s+(?:site|site\s+photo)?)\b", re.I)

COMPLIANCE_PATTERN = re.compile(
    r"\b(wear|wearing|worn|without|not\s+wearing|no\s+helmet|bare\s*head|"
    r"compliance|compliant|violation|violating|unsafe|safe)\b", re.I)

COUNT_PATTERN = re.compile(r"\b(how\s+many|count|number\s+of|total)\b", re.I)

PRESENCE_PATTERN = re.compile(r"\b(is\s+there|are\s+there|any|anyone|does\s+it\s+show)\b", re.I)

# Questions that are plainly not about this image at all.
NON_VISUAL_PATTERN = re.compile(
    r"\b(who\s+(?:are|is)\s+you|what\s+model|your\s+(?:name|architecture|training)|"
    r"capital\s+of|weather\s+(?:today|tomorrow)|what\s+time|tell\s+me\s+a\s+joke|"
    r"how\s+do\s+i\s+(?:train|install|deploy)|what\s+is\s+the\s+meaning|"
    r"translate|write\s+(?:me\s+)?(?:a|some)\s+(?:poem|code|essay))\b", re.I)

# Visual attributes we can see the question is about but genuinely cannot measure.
OUT_OF_SCOPE_VISUAL = re.compile(
    r"\b(colou?r|brand|logo|text|sign\s+say|read\s+the|weather|time\s+of\s+day|"
    r"temperature|age|gender|name\s+of\s+(?:the\s+)?(?:person|worker)|emotion|"
    r"vehicle|truck|crane|ladder|scaffold|glove|vest|boot|goggle|mask)\b", re.I)

KIND_COUNT = "count"
KIND_COMPLIANCE = "compliance"
KIND_PRESENCE = "presence"
KIND_SUMMARY = "summary"
KIND_NOT_IMAGE = "not_about_the_image"
KIND_OUT_OF_SCOPE = "out_of_detector_scope"


@dataclass
class Route:
    needs_detection: bool
    kind: str
    rationale: str
    decided_by: str = "rules"


def route(question: str) -> Route:
    """Decide whether the detector has to run.

    Deterministic by design. Routing is a cheap, high-traffic decision with a
    small, closed vocabulary, so a rule set is both faster and easier to defend
    than a model call, and it cannot hallucinate a route.
    """
    q = (question or "").strip()
    if not q:
        return Route(False, KIND_NOT_IMAGE, "the question was empty")

    if NON_VISUAL_PATTERN.search(q):
        return Route(False, KIND_NOT_IMAGE,
                     "the question asks about something other than the contents "
                     "of the image, so running the detector would not inform it")

    if OUT_OF_SCOPE_VISUAL.search(q) and not COMPLIANCE_PATTERN.search(q):
        return Route(False, KIND_OUT_OF_SCOPE,
                     "the question is about the image but asks for an attribute "
                     "this detector does not predict, so no amount of detection "
                     "would answer it")

    tokens = set(re.findall(r"[a-z-]+", q.lower()))
    mentions_known = bool(tokens & KNOWN_VOCABULARY)
    mentions_image = bool(IMAGE_REFERENCE.search(q))

    if not mentions_known and not mentions_image:
        return Route(False, KIND_NOT_IMAGE,
                     "the question names nothing this detector can see and does "
                     "not refer to the image")

    if COMPLIANCE_PATTERN.search(q):
        return Route(True, KIND_COMPLIANCE,
                     "the question is about who is or is not wearing a helmet, "
                     "which needs per-person detections")
    if COUNT_PATTERN.search(q):
        return Route(True, KIND_COUNT,
                     "the question asks for a count of objects in the image")
    if PRESENCE_PATTERN.search(q):
        return Route(True, KIND_PRESENCE,
                     "the question asks whether something is present in the image")
    return Route(True, KIND_SUMMARY,
                 "the question is about the contents of the image in general")


# ---------------------------------------------------------------------------
# Confidence guardrail
# ---------------------------------------------------------------------------

@dataclass
class Guard:
    sufficient: bool
    reasons: List[str] = field(default_factory=list)


def guardrail(kind: str, facts) -> Guard:
    """Decide whether the detections support an honest answer.

    Runs before the language model and never consults it.
    """
    reasons = []

    if not facts.detections:
        reasons.append("the detector returned no objects above the confidence "
                       "floor of {:.2f}".format(facts.confidence_floor))
        return Guard(False, reasons)

    if facts.max_confidence < LOW_CONFIDENCE_FLOOR:
        reasons.append("every detection scored below {:.2f}, which is too weak to "
                       "assert anything about this image".format(LOW_CONFIDENCE_FLOOR))

    if kind == KIND_COMPLIANCE:
        if not facts.workers:
            reasons.append("no people, helmets or bare heads were detected, so "
                           "helmet compliance cannot be assessed")
        if facts.unknown > 0:
            reasons.append(
                "{} of {} detected people have no helmet and no bare head "
                "associated with them, most likely because their head is occluded, "
                "cropped or too small to resolve".format(
                    facts.unknown, len(facts.workers)))
        headgear_conf = [w.headgear_confidence for w in facts.workers
                         if w.headgear_confidence is not None]
        if headgear_conf and max(headgear_conf) < LOW_CONFIDENCE_FLOOR:
            reasons.append("every helmet and bare-head detection scored below {:.2f}, "
                           "which is too weak to state who is wearing what".format(
                               LOW_CONFIDENCE_FLOOR))

    if kind == KIND_COUNT and facts.mean_confidence < LOW_CONFIDENCE_FLOOR:
        reasons.append("the mean detection confidence of {:.2f} is too low for the "
                       "count to be reliable".format(facts.mean_confidence))

    return Guard(not reasons, reasons)


# ---------------------------------------------------------------------------
# Answer composition
# ---------------------------------------------------------------------------

PEOPLE_WORDS = re.compile(
    r"\b(people|person|persons|workers?|humans?|men|women|man|woman|crew|staff)\b", re.I)


def _deterministic_answer(question: str, kind: str, facts) -> str:
    counts = facts.counts
    if kind == KIND_COUNT:
        if PEOPLE_WORDS.search(question) and facts.workers:
            n = len(facts.workers)
            tail = "{} wearing a helmet, {} bare-headed".format(facts.compliant, facts.violations)
            if facts.unknown:
                tail += ", {} undetermined".format(facts.unknown)
            return "I count {} {} in this image ({}).".format(
                n, "person" if n == 1 else "people", tail)
        parts = ["{} {}".format(v, k if v == 1 else k + "s") for k, v in sorted(counts.items())]
        return "I detect " + (", ".join(parts) if parts else "nothing") + " in this image."
    if kind == KIND_COMPLIANCE:
        return ("Of {} people detected, {} are wearing a helmet and {} are not."
                .format(len(facts.workers), facts.compliant, facts.violations))
    if kind == KIND_PRESENCE:
        present = [k for k, v in counts.items() if v > 0]
        return ("The image contains " + ", ".join(sorted(present)) + "."
                if present else "I detect none of the objects I am trained on.")
    parts = ["{} {}".format(v, k) for k, v in sorted(counts.items())]
    return ("This looks like a work-site image containing " + ", ".join(parts) + ". "
            "{} of the {} detected people are wearing a helmet."
            .format(facts.compliant, len(facts.workers)))


SYSTEM_PROMPT = """You answer questions about a construction-site photograph.

You cannot see the photograph. You are given the structured output of an
RT-DETR object detector that was fine-tuned on three classes: helmet, head
(a human head with no helmet on it), and person. A separate deterministic step
has already matched headgear to people and has already decided that these
detections are sufficient to answer.

Rules:
- Answer only from the structured facts given. Never infer objects, attributes,
  or context that is not in them.
- Be direct and plain. Two or three sentences at most.
- Give the numbers that matter and say what they are counts of.
- Never describe your own confidence as a percentage. State what was detected.
- If the facts contain workers with status "unknown", say how many and that
  their status could not be determined."""


def _llm_answer(question: str, kind: str, facts) -> Optional[str]:
    api_key = os.getenv("ANTHROPIC_API_KEY")
    if not api_key:
        return None
    try:
        import json

        import anthropic
        from pydantic import BaseModel

        class Answer(BaseModel):
            answer: str

        client = anthropic.Anthropic()
        payload = json.dumps(facts.to_dict(), indent=2, sort_keys=True)
        response = client.messages.parse(
            model=MODEL_ID,
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            output_config={"effort": "low"},
            messages=[{
                "role": "user",
                "content": ("Question: {}\nQuestion type: {}\n\nDetector facts:\n{}"
                            .format(question, kind, payload)),
            }],
            output_format=Answer,
        )
        if response.stop_reason == "refusal":
            log.warning("model declined to answer, falling back to templates")
            return None
        return response.parsed_output.answer.strip()
    except Exception as exc:
        log.warning("language model step failed (%s), falling back to templates", exc)
        return None


def compose(question: str, kind: str, facts) -> tuple:
    """Return (answer_text, source) where source is 'llm' or 'template'."""
    text = _llm_answer(question, kind, facts)
    if text:
        return text, "llm"
    return _deterministic_answer(question, kind, facts), "template"


def insufficient_message(kind: str, reasons: List[str]) -> str:
    lead = "I do not have enough information to answer that confidently."
    if kind == KIND_NOT_IMAGE:
        lead = ("That question is not about the contents of the image, so I did "
                "not run the detector.")
    elif kind == KIND_OUT_OF_SCOPE:
        lead = ("I cannot answer that. This detector only recognises people, "
                "helmets and bare heads, so the attribute you asked about is "
                "outside what it can measure.")
    if not reasons:
        return lead
    return lead + " " + " ".join(r[0].upper() + r[1:] + "." for r in reasons)

## 4. Datasets

Downloaded by code, no manual attachment. Both are public Kaggle datasets:

- `andrewmvd/hard-hat-detection` — 5,000 images, Pascal VOC XML. **Training, validation, in-domain test.**
- `mugheesahmad/sh17-dataset-for-ppe-detection` — 8,099 stock-photo images, CC BY-NC-SA 4.0. **Out-of-distribution test only, never trained on.**

In [ ]:
import kagglehub
from pathlib import Path

HARDHAT_ROOT = Path(kagglehub.dataset_download("andrewmvd/hard-hat-detection"))
SH17_ROOT    = Path(kagglehub.dataset_download("mugheesahmad/sh17-dataset-for-ppe-detection"))

print("hard-hat root:", HARDHAT_ROOT)
print("  images:", len(list((HARDHAT_ROOT / "images").glob("*"))) if (HARDHAT_ROOT / "images").is_dir() else "images/ not found")
print("  annotations:", len(list((HARDHAT_ROOT / "annotations").glob("*.xml"))) if (HARDHAT_ROOT / "annotations").is_dir() else "annotations/ not found")
print("sh17 root:", SH17_ROOT)
for p in sorted(SH17_ROOT.iterdir()):
    print("  ", p.name, "(dir)" if p.is_dir() else f"{p.stat().st_size/1e3:.0f} KB")

## 5. Prepare the training data

Pascal VOC → YOLO, with a deterministic `md5(file stem)` image-level split so reruns never leak train images into test.

In [ ]:
!python scripts/prepare_data.py --root "{HARDHAT_ROOT}" --out "{DATA}" --val-frac 0.15 --test-frac 0.15

## 6. Train

`rtdetr-l.pt` is fetched automatically by Ultralytics on first use. AMP is on
(required to fit batch 8), image caching is off (Kaggle RAM is tighter than its
disk). A `training_receipt.json` with hardware, hyperparameters and wall-clock
time is written next to the weights.

In [ ]:
import shutil
if Path(RUNS, RUN_NAME).exists():
    shutil.rmtree(Path(RUNS, RUN_NAME))   # a leftover run must never be mistaken for this one
    print("removed stale run directory", Path(RUNS, RUN_NAME))

!python scripts/train.py \
    --data "{DATA}/data.yaml" \
    --model rtdetr-l.pt \
    --epochs {RUN_EPOCHS} --batch {BATCH} --imgsz {IMGSZ} --lr0 {LR0} --optimizer AdamW \
    --patience 12 --workers {WORKERS} --seed {SEED} --device {DEVICE} --cache {CACHE} \
    --project "{RUNS}" --name {RUN_NAME}

In [ ]:
assert Path(BEST).exists(), f"training did not produce {BEST}"
receipt_path = Path(RUNS) / RUN_NAME / "training_receipt.json"
if not receipt_path.exists():
    # train.py did not reach its receipt step (e.g. the DDP parent crashed after
    # the workers had already saved best.pt). Rebuild it from Ultralytics' own files.
    import csv, yaml
    run_dir = Path(RUNS) / RUN_NAME
    rows = list(csv.DictReader(open(run_dir / "results.csv"))) if (run_dir / "results.csv").exists() else []
    train_args = yaml.safe_load(open(run_dir / "args.yaml")) if (run_dir / "args.yaml").exists() else {}
    last = {k.strip(): v.strip() for k, v in rows[-1].items()} if rows else {}
    receipt = {
        "status": "reconstructed",
        "note": "train.py exited before writing the receipt; values below come from results.csv and args.yaml",
        "hardware": {"gpu": [torch.cuda.get_device_name(i) for i in range(N_GPU)], "torch": torch.__version__},
        "model": train_args.get("model"), "epochs": train_args.get("epochs"), "epochs_completed": len(rows),
        "batch": train_args.get("batch"), "imgsz": train_args.get("imgsz"), "lr0": train_args.get("lr0"),
        "optimizer": train_args.get("optimizer"), "seed": train_args.get("seed"), "device": str(train_args.get("device")),
        "wall_clock_seconds": float(last.get("time", 0)) or None,
        "final_val_mAP50": last.get("metrics/mAP50(B)"), "final_val_mAP50_95": last.get("metrics/mAP50-95(B)"),
        "weights": BEST,
    }
    receipt_path.write_text(json.dumps(receipt, indent=2))
    print("WARNING: receipt was reconstructed; check the end of the training cell output for the error.\n")
print(receipt_path.read_text())
if DRY_RUN:
    print("\nDRY_RUN is on. Read seconds/epoch above, set DRY_RUN = False and EPOCHS, then Save & Run All.")

## 7. Out-of-distribution test set (SH17)

SH17 has 17 classes; only `person`, `head` and `helmet` are kept and remapped
onto this model's class ids. The class order is read from the dataset's own
yaml when it ships one; otherwise the published SH17 order is used as a
fallback and the index map is printed so it can be checked by eye.

In [ ]:
SH17_PUBLISHED_ORDER = ["person", "head", "face", "glasses", "face-mask-medical", "face-guard",
                        "ear", "earmuffs", "hands", "gloves", "foot", "shoes", "safety-vest",
                        "tools", "helmet", "medical-suit", "safety-suit"]

has_yaml = any(True for p in list(SH17_ROOT.rglob("*.yaml")) + list(SH17_ROOT.rglob("*.yml"))
               if "names" in p.read_text(encoding="utf-8", errors="replace"))
names_arg = ""
if not has_yaml:
    fallback = Path(WORK) / "sh17_names.yaml"
    fallback.write_text("names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(SH17_PUBLISHED_ORDER)))
    names_arg = f'--names "{fallback}"'
    print("no yaml with a names: block in the dataset; using the published SH17 class order")

!python scripts/prepare_ood.py --root "{SH17_ROOT}" --out "{OOD}" --limit {OOD_LIMIT} {names_arg}

## 8. Evaluate

mAP / precision / recall per class on the in-domain test split and on SH17. The gap between the two is the dataset-specific part of the in-domain score.

In [ ]:
!python scripts/evaluate.py \
    --weights "{BEST}" \
    --data "{DATA}/data.yaml" \
    --ood  "{OOD}/data.yaml" \
    --imgsz {IMGSZ} --batch {BATCH} \
    --out  "{ARTS}/metrics.json"

## 9. Failure mining

Worst test images by error count, each miss measured for blur, scale, occlusion and exposure. Confirm every hypothesis by looking at the annotated image before it goes in the memo.

In [ ]:
!python scripts/failure_cases.py \
    --weights "{BEST}" \
    --data "{DATA}/data.yaml" \
    --top 8 \
    --out "{ARTS}/failures"

In [ ]:
import matplotlib.pyplot as plt
import cv2

report = json.load(open(f"{ARTS}/failures/failure_report.json"))
worst = report["worst"][:4]
fig, axes = plt.subplots(max(len(worst), 1), 1, figsize=(10, 6 * max(len(worst), 1)))
for ax, rec in zip(axes if len(worst) > 1 else [axes], worst):
    img = cv2.cvtColor(cv2.imread(f"{ARTS}/failures/failure_{rec['image']}"), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.axis("off")
    hyp = "; ".join(h for e in rec["errors"] for h in e["hypotheses"])
    ax.set_title(f"{rec['image']} — {rec['error_count']} errors\n{hyp[:160]}", fontsize=9)
plt.tight_layout(); plt.show()

## 10. End-to-end check of the reasoning layer

Runs the same code path as the API's `POST /ask` — route → detect → associate →
guardrail → compose — on a test image, offline. No API key is needed; without
one the answer comes from templates over the same structured facts.

In [ ]:
os.environ["MODEL_WEIGHTS"] = BEST
sys.path.insert(0, "/kaggle/working/repo")
import importlib
for m in ("app.detector", "app.scene", "app.reasoning"):
    if m in sys.modules: importlib.reload(sys.modules[m])
from app.detector import Detector
from app.scene import build_scene
from app import reasoning

det = Detector(weights=BEST, imgsz=IMGSZ)
sample = sorted(Path(f"{DATA}/images/test").glob("*"))[0]
image_bytes = sample.read_bytes()

for question in ["Is anyone not wearing a helmet?",
                 "How many people are in this image?",
                 "What colour is the truck?",
                 "What is the capital of France?"]:
    decision = reasoning.route(question)
    print("\nQ:", question)
    print("   route:", decision.kind, "| detector needed:", decision.needs_detection)
    if not decision.needs_detection:
        print("   A:", reasoning.insufficient_message(decision.kind, []))
        continue
    dets, w, h, ms = det.predict(image_bytes, conf=0.25)
    facts = build_scene(dets, w, h, 0.25)
    guard = reasoning.guardrail(decision.kind, facts)
    if not guard.sufficient:
        print("   A:", reasoning.insufficient_message(decision.kind, guard.reasons))
    else:
        answer, source = reasoning.compose(question, decision.kind, facts)
        print(f"   A ({source}):", answer)
    print("   evidence:", {k: facts.to_dict()[k] for k in ("counts", "compliant_workers", "violations", "undetermined_workers")})

## 11. Package the artifacts

Everything the API and the memo need lands in `/kaggle/working/artifacts`,
plus a single zip. Download from the notebook's **Output** tab, then:

- `best.pt` -> `artifacts/best.pt` in the repo
- `samples/` -> `samples/` in the repo (the README curl examples use them)
- `metrics.json`, `training_receipt.json`, `split_stats.json`, `failures/` -> fill the memo

then start the API with `uvicorn app.main:app --host 0.0.0.0 --port 8000`.

In [ ]:
import shutil
shutil.copy(BEST, f"{ARTS}/best.pt")
shutil.copy(f"{RUNS}/{RUN_NAME}/weights/last.pt", f"{ARTS}/last.pt")
shutil.copy(f"{RUNS}/{RUN_NAME}/training_receipt.json", ARTS)
shutil.copy(f"{DATA}/split_stats.json", ARTS)
if Path(f"{OOD}/ood_stats.json").exists():
    shutil.copy(f"{OOD}/ood_stats.json", ARTS)
for plot in ("results.png", "confusion_matrix.png", "PR_curve.png"):
    src = Path(RUNS) / RUN_NAME / plot
    if src.exists():
        shutil.copy(src, ARTS)

# Two real test images so the README curl examples work on a clean checkout.
os.makedirs(f"{ARTS}/samples", exist_ok=True)
for i, img in enumerate(sorted(Path(f"{DATA}/images/test").glob("*"))[:2], 1):
    shutil.copy(img, f"{ARTS}/samples/site_{i:02d}{img.suffix}")

shutil.make_archive(f"{WORK}/ppe_artifacts", "zip", ARTS)
for p in sorted(Path(ARTS).rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size/1e6:8.1f} MB  {p.relative_to(ARTS)}")
print(f"\nzip: {WORK}/ppe_artifacts.zip  {Path(f'{WORK}/ppe_artifacts.zip').stat().st_size/1e6:.1f} MB")